<a href="https://colab.research.google.com/github/lsgrep/serv/blob/main/notebooks/04_qlora_oom_postmortem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4 — QLoRA on a T4, and the OOM postmortem

**The claim you should be able to make when you finish:** *"I can budget
training memory before I start, and when it OOMs anyway I can show you the
allocator snapshot and point at the tensor that did it."*

Anyone can hit an OOM. The skill being tested is what you do in the next five
minutes — and the answer is not "lower the batch size and rerun". It is: dump
the allocator history, look at what was live at the moment of failure, and fix
the term that actually dominates.

QLoRA on a T4 is the canonical Colab workload — it is what QLoRA was pitched on
— so this lab is a natural fit for the free tier.

### The plan

1. Budget the run on paper: weights, gradients, optimizer, activations.
2. Wrap the training loop so an OOM dumps a memory snapshot instead of a traceback.
3. Deliberately overshoot. Read the snapshot. Name the term that broke it.
4. Fix it with the cheapest lever that works, and finish a real (small) run.

In [ ]:
# Cell 1 — idempotent bootstrap. The installs are the slow part; do them once.
REPO   = "https://github.com/lsgrep/serv.git"
BRANCH = "main"

import os, subprocess, sys

if not os.path.isdir("serv"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], check=True)
else:
    subprocess.run(["git", "-C", "serv", "pull", "--ff-only", "-q"], check=False)
sys.path.insert(0, os.path.abspath("serv"))

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "matplotlib", "transformers", "peft", "bitsandbytes",
                "trl", "datasets", "accelerate"], check=True)

import servlab
env = servlab.notebook_setup()

## 1. Budget it on paper

Training memory has four terms, and full fine-tuning is dominated by the one
people forget:

| term | full fine-tune (7B, fp16) | QLoRA |
|---|---|---|
| weights | ~14 GiB | ~4 GiB (NF4) |
| gradients | ~14 GiB | ~10 MiB (adapters only) |
| optimizer (AdamW: 2 moments + fp32 master) | ~84 GiB | ~20 MiB |
| activations | batch × seq dependent | **unchanged** |

QLoRA collapses three of the four. It does **nothing** for activations — those
scale with `batch × seq_len × hidden × layers` regardless of how the weights are
stored. So on a 16 GB card, a QLoRA run OOMs on *sequence length*, not on model
size, and that is the thing to say out loud when someone asks why your run died.

In [ ]:
from servlab.memory import training_budget, lora_trainable_params, GIB

# Llama-3.2-3B shapes. Swap in whatever you are actually fine-tuning.
PARAMS, HIDDEN, LAYERS = 3.21e9, 3072, 28
LORA_RANK, BATCH, SEQ = 16, 1, 1024

adapters = lora_trainable_params(PARAMS, hidden=HIDDEN, n_layers=LAYERS,
                                 rank=LORA_RANK, targets=4)
print(f"LoRA adapters: {adapters/1e6:.1f}M trainable params "
      f"({adapters/PARAMS:.2%} of the model)\n")

print("full fine-tune, fp16 + AdamW:")
print(training_budget(PARAMS, weight_bits=16, optimizer="adamw", batch=BATCH,
                      seq_len=SEQ, n_layers=LAYERS, hidden=HIDDEN))
print("\nQLoRA: 4-bit frozen weights, adapters in fp16, paged 8-bit AdamW:")
print(training_budget(PARAMS, weight_bits=4, trainable_params=adapters,
                      optimizer="adamw8bit", batch=BATCH, seq_len=SEQ,
                      n_layers=LAYERS, hidden=HIDDEN))
print(f"\nT4 has {env.vram_gb:.1f} GiB.")

In [ ]:
# Activations are the term QLoRA does not touch. Here is the shape of that.
import matplotlib.pyplot as plt
from servlab.plots import use_style, SERIES, STATUS

use_style()
seqs = [256, 512, 1024, 2048, 4096, 8192]
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for i, (label, ckpt) in enumerate([("no checkpointing", False), ("gradient checkpointing", True)]):
    ys = [training_budget(PARAMS, weight_bits=4, trainable_params=adapters,
                          optimizer="adamw8bit", batch=1, seq_len=s, n_layers=LAYERS,
                          hidden=HIDDEN, activation_checkpointing=ckpt).total / GIB
          for s in seqs]
    ax.plot(seqs, ys, marker="o", color=SERIES[i], label=label)
ax.axhline(env.vram_gb or 16, color=STATUS["critical"], linestyle=":", linewidth=1.5)
ax.annotate(f"{env.vram_gb or 16:.0f} GiB card", xy=(seqs[0], env.vram_gb or 16),
            xytext=(0, 5), textcoords="offset points", color=STATUS["critical"], fontsize=9)
ax.set_xscale("log", base=2)
ax.set_xlabel("sequence length"); ax.set_ylabel("predicted peak (GiB)")
ax.set_title("QLoRA memory is a sequence-length problem")
ax.legend(loc="upper left")
plt.show()

print("Pick the sequence length where the prediction crosses the line — that is")
print("where you expect the OOM. Writing the prediction down first is the lab.")

## 2. Load the model in 4-bit

NF4 with double quantisation, compute in fp16 (not bf16 — Turing has neither
bf16 nor the tensor cores for it, and `bnb_4bit_compute_dtype=torch.bfloat16`
will either error or silently fall back).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from servlab.memory import gpu_memory_report, reset_peak

BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"   # gated: needs a HF token
# Ungated alternatives if you would rather not authenticate:
# BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
# BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

COMPUTE_DTYPE = torch.bfloat16 if env.supports_bf16 else torch.float16
print(f"compute dtype: {COMPUTE_DTYPE}  (bf16 available: {env.supports_bf16})")

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # normal-float 4, not plain int4
    bnb_4bit_use_double_quant=True,     # quantise the quantisation constants too
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                             device_map={"": 0})
model.config.use_cache = False           # incompatible with gradient checkpointing
reset_peak()
print(gpu_memory_report())

In [ ]:
# How close was the weights prediction?
from servlab.memory import GIB
import torch

predicted = training_budget(PARAMS, weight_bits=4, trainable_params=0,
                            optimizer="none", batch=0, seq_len=0,
                            n_layers=LAYERS, hidden=HIDDEN, overhead_gb=0).weights
print(f"predicted 4-bit weights  {predicted/GIB:.2f} GiB")
print(f"measured allocated       {torch.cuda.memory_allocated()/GIB:.2f} GiB")
print("\nThe measurement runs higher: NF4 stores per-block scales, the embedding")
print("and lm_head usually stay in higher precision, and the allocator rounds up.")

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
peft_cfg = LoraConfig(
    r=LORA_RANK, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, peft_cfg)
model.print_trainable_parameters()

## 3. Make the OOM produce evidence

`torch.cuda.memory._record_memory_history()` records every allocation with the
stack that made it. `_dump_snapshot()` writes it to a pickle you drag onto
[pytorch.org/memory_viz](https://pytorch.org/memory_viz), which draws memory
over time with each block attributable to the code that allocated it.

`servlab.memory.record_memory` is a context manager that turns it on, and dumps
on the way out **if the block raised** — which is exactly when you want it and
exactly when you will forget to do it by hand.

Now overshoot on purpose. Pick a sequence length past the crossing point in the
chart above.

In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

data = load_dataset("tatsu-lab/alpaca", split="train[:400]")

def to_text(ex):
    prompt = ex["instruction"] + (("\n\n" + ex["input"]) if ex["input"] else "")
    return {"text": f"<|user|>\n{prompt}\n<|assistant|>\n{ex['output']}"}

data = data.map(to_text, remove_columns=data.column_names)
print(data[0]["text"][:220])

In [ ]:
# Deliberately too big. Raise OOM_SEQ / OOM_BATCH until it fails if it survives.
OOM_SEQ, OOM_BATCH = 4096, 4

from servlab.memory import record_memory, oom_hints, reset_peak, gpu_memory_report

reset_peak()
args = SFTConfig(
    output_dir="runs/qlora-oom",
    per_device_train_batch_size=OOM_BATCH,
    gradient_accumulation_steps=1,
    max_length=OOM_SEQ,
    max_steps=8,
    learning_rate=2e-4,
    fp16=not env.supports_bf16,
    bf16=env.supports_bf16,
    gradient_checkpointing=False,     # off on purpose: activations are the point
    optim="paged_adamw_8bit",
    logging_steps=1,
    report_to=[],
)

try:
    with record_memory("runs/oom_snapshot.pickle"):
        SFTTrainer(model=model, args=args, train_dataset=data).train()
    print("no OOM — raise OOM_SEQ or OOM_BATCH and run again")
except torch.cuda.OutOfMemoryError as exc:
    print(oom_hints(exc))

In [ ]:
# Get the snapshot onto your laptop and open https://pytorch.org/memory_viz
try:
    from google.colab import files
    files.download("runs/oom_snapshot.pickle")
except Exception as exc:
    print("not in Colab, or download blocked:", exc)
    print("the file is at runs/oom_snapshot.pickle")

### Reading the snapshot

On [pytorch.org/memory_viz](https://pytorch.org/memory_viz), drag the pickle in
and look at **Active Memory Timeline**. Work through it in this order:

1. **Find the wall** — the point where total allocation stops rising because it
   cannot. What was the last thing allocated?
2. **Look at the block sizes.** A few enormous blocks means one tensor is
   miscalculated (usually an attention score matrix at long sequence length). A
   sawtooth of many medium blocks means activations across layers.
3. **Check the flat band at the bottom.** That is the 4-bit weights, and they
   are not your problem — if that band is most of the card, you picked too big a
   model, which is a different fix.
4. **Compare `reserved` against `allocated`.** A big gap is fragmentation: the
   allocator holds memory the driver gave it but cannot place your tensor in.
   `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` is the fix for that, and
   *only* for that — it does nothing if you genuinely need more memory than
   exists.

The sentence to be able to say: *"the snapshot showed activations for a
4096-token batch of 4, about N GiB, which is what the budget predicted; the
weights were never the problem."*

In [ ]:
print(gpu_memory_report())
print()
print("allocated  = live tensors right now")
print("reserved   = what the caching allocator holds from the driver")
print("the gap    = fragmentation; an OOM can happen with GiB apparently free")

## 4. Fix it, cheapest lever first

In order of what you should reach for:

1. **`per_device_train_batch_size=1`** with `gradient_accumulation_steps` raised
   to keep the effective batch. Free, and usually enough.
2. **`gradient_checkpointing=True`** — recompute activations in the backward
   pass instead of storing them. Costs ~30% step time, removes most of the
   activation term. This is the big one for QLoRA.
3. **Shorter `max_length`** — activations are linear in it. Look at your actual
   token-length distribution first; people commonly pad to 4x their p99.
4. **`paged_adamw_8bit`** — already on. Optimizer state pages to host memory
   under pressure instead of OOMing.
5. **Lower LoRA rank / fewer target modules** — surprisingly little memory, real
   quality cost. Late in the list for a reason.

Restart the session before this cell: the failed run left fragmented memory
behind, and you want a clean allocator, not a confound.

In [ ]:
# After a session restart, rerun cell 1 and the model-loading cells, then this.
FIXED_SEQ, FIXED_BATCH, ACCUM = 1024, 1, 8

reset_peak()
args = SFTConfig(
    output_dir="runs/qlora-fixed",
    per_device_train_batch_size=FIXED_BATCH,
    gradient_accumulation_steps=ACCUM,          # same effective batch, a fraction of the memory
    max_length=FIXED_SEQ,
    max_steps=30,
    learning_rate=2e-4,
    fp16=not env.supports_bf16,
    bf16=env.supports_bf16,
    gradient_checkpointing=True,                # the lever that mattered
    optim="paged_adamw_8bit",
    logging_steps=5,
    save_steps=15,                              # checkpoint: Colab disconnects
    report_to=[],
)

trainer = SFTTrainer(model=model, args=args, train_dataset=data)
with record_memory("runs/fixed_snapshot.pickle", dump_always=True):
    trainer.train()

print("\n" + gpu_memory_report())

In [ ]:
# Predicted vs measured peak. Being within ~30% means the mental model works.
import torch
from servlab.memory import GIB

pred = training_budget(PARAMS, weight_bits=4, trainable_params=adapters,
                       optimizer="adamw8bit", batch=FIXED_BATCH, seq_len=FIXED_SEQ,
                       n_layers=LAYERS, hidden=HIDDEN, activation_checkpointing=True)
print(pred)
print(f"\n  measured peak              {torch.cuda.max_memory_allocated()/GIB:7.2f} GiB")
print(f"  ratio measured/predicted   {torch.cuda.max_memory_allocated()/pred.total:7.2f}")

In [ ]:
# Save the adapter (a few MB) to Drive — the session will not survive the night.
from servlab.env import mount_drive

dest = "/content/drive/MyDrive/servlab/qlora-adapter" if mount_drive() else "runs/qlora-adapter"
trainer.model.save_pretrained(dest)
tok.save_pretrained(dest)
print("adapter ->", dest)

In [ ]:
# Did it learn anything? A qualitative check, not an eval — lab 5 does evals.
from transformers import pipeline

pipe = pipeline("text-generation", model=trainer.model, tokenizer=tok, max_new_tokens=96)
print(pipe("<|user|>\nExplain what a KV cache is in two sentences.\n<|assistant|>\n")[0]["generated_text"])

## What to be able to say afterwards

1. **The four terms of training memory**, and which ones QLoRA removes (three)
   and which it does not (activations).
2. **Why a QLoRA OOM is usually a sequence-length problem**, and why "use a
   smaller model" is often the wrong response.
3. **How to get evidence**: `_record_memory_history` before, `_dump_snapshot` in
   the failure path, memory_viz to read it.
4. **The difference between fragmentation and genuine demand**, and that
   `expandable_segments` only helps the first.
5. **The fix order**, cheapest first — and that gradient checkpointing is a
   *time for memory* trade you should be able to quantify (~30%), not a magic flag.

**Next:** lab 5 asks whether the quantisation trick that made this fit costs you
any quality.